# Ghost imaging with a LINCam

A photon pair is born in a PPKTP crystal. The visible **signal** photon is imaged
onto the LINCam and never meets the sample; the infrared **idler** goes through
the sample onto a single-pixel "bucket" SPAD that sees no picture at all. The
SPAD's pulse drives the LINCam's reference input, so every photon in the file carries
its own arrival time and that of the last bucket click before it, both in
picoseconds.

The setup is that of Ryan *et al.*, [Optica 11, 1261 (2024)](https://doi.org/10.1364/OPTICA.527982),
with a LINCam where the paper has its NCam.

Three steps, all in [`ghost.py`](ghost.py), which this notebook only calls:

1. histogram every photon: the **full image**, the bare crystal face;
2. histogram `start - stop`: the **coincidence peak** over a floor of accidentals;
3. keep the photons under the peak: the **ghost image** of a sample the camera never saw.

The recorded `/photons/dt` holds the same delay in 100 ps channels and is not
used: the peak is narrower than two of them.

In [ ]:
import numpy as np
from bokeh.io import output_notebook, show
from bokeh.layouts import row
from photonscore import Vault

import ghost

output_notebook()

FILE = "ghost-dataset.photons"
BINS = 512   # pixels a side: 512, 1024 or 2048
vault = Vault(FILE)
print(vault)

## 1. Every photon

One pass over the file, in chunks, fills both the image and the delay histogram
(1 ps a bin). The image is drawn the way Photonscore Preview draws it.

In [ ]:
full, delay_counts = ghost.scan(vault, bins=BINS)
print(f"{int(full.sum()):,} photons")
show(ghost.image_figure(full, "Every photon"))

## 2. The coincidence peak

A signal photon whose twin reached the bucket arrives a fixed delay after the
click: cable lengths and optical paths, the same for every pair. Everything
else is an accidental and spreads flat.

In [ ]:
peak_ps, fwhm_ps, floor = ghost.find_peak(delay_counts)
print(f"peak at {peak_ps} ps, {fwhm_ps} ps FWHM, {floor:.1f} accidentals per ps")

WINDOW_PS = 300
show(ghost.delay_figure(delay_counts, peak_ps, WINDOW_PS, fwhm_ps))
show(ghost.delay_figure(delay_counts, peak_ps, WINDOW_PS, fwhm_ps, around_ps=1000))

How wide to cut is a trade: a wider window keeps more true pairs and lets in
more accidentals. The delay histogram already holds the answer, no further pass
needed.

In [ ]:
print(f"{'window':>10} {'photons':>12} {'accidental':>11}")
for w in (25, 50, 100, 150, 250, 500):
  kept = int(delay_counts[max(0, peak_ps - w):peak_ps + w + 1].sum())
  print(f"{'+/- ' + str(w) + ' ps':>10} {kept:>12,} {100 * floor * (2 * w + 1) / kept:>10.2f}%")

## 3. The ghost image

A second pass keeps the photons within the window. Set `SAVE` to a file name to
also write them out as a `.photons` file: Preview opens it, and so does this
notebook. Its `/photons/dt` is the TDC delay at 1 ps a channel.

In [ ]:
SAVE = None   # e.g. "ghost-only.photons"

ghost_image, selected = ghost.select(vault, peak_ps, WINDOW_PS, bins=BINS, save=SAVE)
print(f"{selected:,} photons within {peak_ps} +/- {WINDOW_PS} ps: "
      f"{100 * selected / full.sum():.2f} % of the file")

show(row(ghost.image_figure(full, "Every photon", size=440),
         ghost.image_figure(ghost_image, f"Peak +/- {WINDOW_PS} ps", size=440)))

## 4. The two pictures as PNGs

8-bit grayscale, a pixel per bin, with the contrast and the orientation of the
figures above: what is white there is white here.

In [ ]:
import os
from IPython.display import Image, display

stem = os.path.splitext(FILE)[0]
for name, picture in ((f"{stem}.full.png", full), (f"{stem}.ghost.png", ghost_image)):
  ghost.save_png(name, picture)
  print(name, f"{os.path.getsize(name):,} bytes")
  display(Image(name))

In [ ]:
vault.close()